# Esta parte do código se refere à pipeline da camada GOLD em BATCH para testes antes de subir ao AWS

In [1]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# pyarrow para salvar em PARQUET

!pip install pyarrow colorama tabulate --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import logging
import time
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

In [3]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
from pathlib import Path
from datetime import datetime

DATA_SILVER = Path("silver")
DATA_GOLD = Path("gold")

DATA_GOLD.mkdir(parents=True, exist_ok=True)

PROCESSAMENTO = datetime.now()

In [4]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)

log = logging.getLogger(__name__)

In [5]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA GOLD")
log.info("~" * 35)

2026-08-18 23:10:19,210 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-18 23:10:19,211 | INFO     | INICIANDO ETL DA CAMADA GOLD
2026-08-18 23:10:19,212 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [6]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# OBSERVABILIDADE: MÉTRICAS ESTRUTURADAS E ALERTAS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# (mesmo padrão usado nas camadas Bronze e Silver)

def log_metrica(evento, **campos):
    """
    Loga um evento estruturado (campo=valor), permitindo consulta via
    CloudWatch Logs Insights, ex.:
        fields @timestamp, tabela, volume, latencia_segundos
        | filter evento = "tabela_processada"
    """
    campos_formatados = " | ".join(f"{chave}={valor}" for chave, valor in campos.items())
    log.info(f"[METRICA] evento={evento} | {campos_formatados}")


def emitir_alerta(mensagem, **contexto):
    """
    Emite um alerta de erro (sempre em nível ERROR no log) e tenta
    publicar em um tópico SNS, se configurado via variável de ambiente
    SNS_TOPIC_ARN, para que a falha não dependa de alguém checar o log
    manualmente.
    """
    contexto_formatado = " | ".join(f"{k}={v}" for k, v in contexto.items())
    log.error(f"[ALERTA] {mensagem} | {contexto_formatado}")

    topico_sns = os.environ.get("SNS_TOPIC_ARN")

    if not topico_sns:
        log.warning("[ALERTA] SNS_TOPIC_ARN não configurado - alerta ficou registrado apenas no log")
        return

    try:
        import boto3
        sns = boto3.client("sns")
        sns.publish(
            TopicArn=topico_sns,
            Subject="[Tech Challenge] Falha no pipeline",
            Message=f"{mensagem}\n\n{contexto_formatado}"
        )
        log.info("[ALERTA] Notificação SNS publicada com sucesso")
    except Exception as e:
        log.warning(f"[ALERTA] Falha ao publicar no SNS (alerta permanece apenas no log): {e}")

In [7]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# LENDO ARQUIVOS DA CAMADA SILVER
"""
    Lê um arquivo Parquet da camada SILVER.

    Args:
        tabela (str): Nome da tabela.
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_silver(tabela):

    caminho = DATA_SILVER / f"{tabela}.parquet"

    log.info(f"Lendo Silver: {caminho}")

    return pd.read_parquet(caminho)

In [8]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_uf():

    log.info("Construindo Gold: Ranking UF")

    df = ler_silver("uf")

    # removendo o Total, pois só há uma única informação
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    # ordena
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()
    
    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking UF criado")

    return df

In [9]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING MUNICÍPIOS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_municipio():

    log.info("Construindo Gold: Ranking Municípios")

    df = ler_silver("municipio")

    # Ordena por ano, rede e taxa
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # Cria ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()

    # Mantém somente as colunas importantes
    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking de Municípios criado")

    return df

In [10]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - EVOLUÇÃO UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def gold_evolucao_uf():

    log.info("Construindo Gold: Evolução UF")

    df = ler_silver("uf")

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "media_portugues"
        ]
    ].copy()

    df["_gold_processed_at"] = datetime.now()

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Evolução UF criada")

    return df

In [11]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RESUMO POR REDE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_resumo_rede():

    log.info("Construindo Gold: Resumo por Rede")

    df = ler_silver("uf")
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ]

    df_gold = (        
        df.groupby(["ano", "rede"])
          .agg(
              media_taxa_alfabetizacao=(
                  "taxa_alfabetizacao",
                  "mean"
              ),
              media_portugues=(
                  "media_portugues",
                  "mean"
              ),
              quantidade_ufs=(
                  "sigla_uf",
                  "nunique"
              )
          )
          .reset_index()
    )
    
    df_gold["media_taxa_alfabetizacao"] = (
    df_gold["media_taxa_alfabetizacao"].round(2)
    )

    df_gold["media_portugues"] = (
        df_gold["media_portugues"].round(2)
    )

    df_gold["_gold_processed_at"] = datetime.now()

    log.info("Resumo por Rede criado")

    return df_gold

In [12]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# HELPER: EXTRAI A META DO PRÓPRIO ANO DA LINHA
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# As tabelas de meta vêm em formato LARGO: uma coluna por ano-alvo
# (meta_alfabetizacao_2024 ... meta_alfabetizacao_2030) na mesma linha
# que carrega a taxa observada (`taxa_alfabetizacao`) daquele ano. Para
# comparar "resultado do ano X" com "meta do ano X", é preciso pegar,
# em cada linha, a coluna de meta cujo ano bate com o `ano` da própria
# linha.

def _extrair_meta_do_ano(df):
    """
    Args:
        df (pandas.DataFrame): Tabela de meta (uf ou município), contendo
            a coluna `ano` e as colunas `meta_alfabetizacao_2024..2030`.

    Returns:
        pandas.Series: Valor da meta definida para o próprio `ano` da
            linha (NaN se não houver meta definida para aquele ano, ex.:
            anos anteriores a 2024).
    """

    colunas_meta = {
        ano: f"meta_alfabetizacao_{ano}"
        for ano in range(2024, 2031)
        if f"meta_alfabetizacao_{ano}" in df.columns
    }

    meta_do_ano = pd.Series(np.nan, index=df.index, dtype="float64")

    for ano, coluna in colunas_meta.items():
        mascara = df["ano"] == ano
        meta_do_ano.loc[mascara] = df.loc[mascara, coluna]

    return meta_do_ano

In [13]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (UF)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_uf():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - UF")

    df = ler_silver("meta_alfabetizacao_uf")

    # removendo o Total, mesmo critério usado no ranking/evolução de UF
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    # só faz sentido comparar quando existe meta definida para aquele ano
    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (UF) criada")

    return df

In [14]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (MUNICÍPIO)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_municipio():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - Município")

    df = ler_silver("meta_alfabetizacao_municipio")

    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "nivel_alfabetizacao",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["id_municipio", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (Município) criada")

    return df

In [15]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE DATA QUALITY
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CHECKS = {

    "gold_ranking_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0,100),
            "critico": True
        }

    ],
    "ranking_municipio": [

        {
            "tipo": "min_count",
            "valor": 1000,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }
    ],
    "evolucao_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }

    ],
    "resumo_rede": [

    {
        "tipo": "min_count",
        "valor": 7,
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "ano",
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "rede",
        "critico": True
    },

    {
        "tipo": "range",
        "coluna": "media_taxa_alfabetizacao",
        "valor": (0, 100),
        "critico": True
    }

],

    "comparacao_meta_uf": [

        {
            # valor conservador: como só existem metas para 2024-2030,
            # o volume real depende de quantos desses anos já têm
            # avaliação (ano) registrada na base. Ajustar após a
            # primeira execução com dados reais.
            "tipo": "min_count",
            "valor": 10,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ],

    "comparacao_meta_municipio": [

        {
            "tipo": "min_count",
            "valor": 50,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ]

}

In [16]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE QUALIDADE DA CAMADA GOLD
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def checar_qualidade(df, checks):

    log.info("Iniciando verificações de qualidade")

    for check in checks:

        if check["tipo"] == "min_count":

            assert len(df) >= check["valor"], \
                f"Quantidade mínima não atendida ({len(df)} registros)."

        elif check["tipo"] == "not_null":

            coluna = check["coluna"]

            assert df[coluna].isnull().sum() == 0, \
                f"Existem valores nulos na coluna '{coluna}'."

        elif check["tipo"] == "range":

            coluna = check["coluna"]
            minimo, maximo = check["valor"]

            assert (
                df[coluna].between(minimo, maximo).all()
            ), f"Valores fora do intervalo na coluna '{coluna}'."

    log.info("Checks de qualidade concluídos com sucesso!")

In [17]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SALVA UM DATAFRAME NA CAMADA GOLD EM FORMATO PARQUET
"""
    Args:
        df (pandas.DataFrame): DataFrame tratado.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def salvar_gold(df, nome):

    caminho = DATA_GOLD / f"{nome}.parquet"

    df.to_parquet(
        caminho,
        index=False
    )

    log.info(f"Camada GOLD salva em {caminho}")

    return caminho

In [18]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# EXECUÇÃO DA CAMADA GOLD
"""
    Cada dataset Gold é construído de forma isolada: se um deles falhar,
    um alerta é emitido e os demais continuam sendo processados.
    Latência e volume de cada dataset são registrados como métricas
    estruturadas e consultáveis.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~
def executar_gold():

    log.info("~" * 35)
    log.info("INICIANDO CAMADA GOLD")
    log.info("~" * 35)

    inicio_pipeline = time.perf_counter()

    # Cada item: (nome do dataset salvo, função que constrói o dataframe, chave em CHECKS)
    datasets = [
        ("ranking_uf", gold_ranking_uf, "gold_ranking_uf"),
        ("ranking_municipio", gold_ranking_municipio, "ranking_municipio"),
        ("evolucao_uf", gold_evolucao_uf, "evolucao_uf"),
        ("resumo_rede", gold_resumo_rede, "resumo_rede"),
        ("comparacao_meta_uf", gold_comparacao_meta_uf, "comparacao_meta_uf"),
        ("comparacao_meta_municipio", gold_comparacao_meta_municipio, "comparacao_meta_municipio"),
    ]

    datasets_ok = 0
    datasets_falha = 0

    for nome, funcao_construtora, chave_checks in datasets:

        log.info(f"Checando qualidade de {nome}")

        inicio_dataset = time.perf_counter()

        try:

            df_gold = funcao_construtora()

            checar_qualidade(
                df_gold,
                CHECKS[chave_checks]
            )

            salvar_gold(
                df_gold,
                nome
            )

            latencia_segundos = round(time.perf_counter() - inicio_dataset, 2)

            log_metrica(
                "tabela_processada",
                camada="gold",
                tabela=nome,
                volume=len(df_gold),
                latencia_segundos=latencia_segundos,
                status="sucesso"
            )

            datasets_ok += 1

        except Exception as e:

            latencia_segundos = round(time.perf_counter() - inicio_dataset, 2)

            log_metrica(
                "tabela_processada",
                camada="gold",
                tabela=nome,
                volume=0,
                latencia_segundos=latencia_segundos,
                status="falha"
            )

            emitir_alerta(
                f"Falha na construção do dataset Gold '{nome}'",
                camada="gold",
                tabela=nome,
                erro=str(e)
            )

            datasets_falha += 1

            # isola a falha: segue para o próximo dataset
            continue

    latencia_total_segundos = round(time.perf_counter() - inicio_pipeline, 2)

    log_metrica(
        "pipeline_concluido",
        camada="gold",
        tabelas_ok=datasets_ok,
        tabelas_falha=datasets_falha,
        latencia_total_segundos=latencia_total_segundos
    )

    if datasets_falha > 0:
        emitir_alerta(
            f"Pipeline Gold concluído com {datasets_falha} falha(s) de {datasets_ok + datasets_falha} dataset(s)",
            camada="gold",
            datasets_falha=datasets_falha,
            datasets_ok=datasets_ok
        )

    log.info("Camada Gold concluída!")

In [19]:
executar_gold()

2026-08-18 23:10:19,379 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-18 23:10:19,380 | INFO     | INICIANDO CAMADA GOLD
2026-08-18 23:10:19,380 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-18 23:10:19,381 | INFO     | Checando qualidade de ranking_uf
2026-08-18 23:10:19,381 | INFO     | Construindo Gold: Ranking UF
2026-08-18 23:10:19,382 | INFO     | Lendo Silver: silver\uf.parquet
2026-08-18 23:10:19,468 | INFO     | Ranking UF criado
2026-08-18 23:10:19,469 | INFO     | Iniciando verificações de qualidade
2026-08-18 23:10:19,470 | INFO     | Checks de qualidade concluídos com sucesso!
2026-08-18 23:10:19,478 | INFO     | Camada GOLD salva em gold\ranking_uf.parquet
2026-08-18 23:10:19,479 | INFO     | [METRICA] evento=tabela_processada | camada=gold | tabela=ranking_uf | volume=144 | latencia_segundos=0.1 | status=sucesso
2026-08-18 23:10:19,480 | INFO     | Checando qualidade de ranking_municipio
2026-08-18 23:10:19,480 | INFO     | Construindo Gold: Ranking